# MultiConAD Benchmarks
The following implementations are cloned from the orignial MultiConAD repo.\
Only minor tweaks are performed for workflow integration.

## Definitions

### Dataset Creation Functions

In [16]:
"""
This is a slightly modified version of the "text_cleaning_English.py" script
from the MultiConAD repo:
https://github.com/ArezoShakeri/MultiConAD
"""

import re
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
from sklearn.model_selection import train_test_split
from collection import JSONLCombiner


# Pitt, Lu, Baycrest, VAS, Kempler, WLS, Delware, taukdial_English_train, taukdial_English_test
input_files = [
    "../data/Pitt/pitt.jsonl",
    "../data/Lu/lu.jsonl",
    "../data/Baycrest/baycrest.jsonl",
    "../data/VAS/vas.jsonl",
    "../data/Kempler/kempler.jsonl",
    "../data/WLS/wls.jsonl",
    "../data/Delaware/delaware.jsonl",
    "../data/Taukdial/taukdial.jsonl",
]
output_directory = '.tmp/'
output_filename = 'English.jsonl'
combiner = JSONLCombiner(input_files, output_directory, output_filename)
combiner.combine()
train_filename = 'train_' + output_filename
test_filename = 'test_' + output_filename

output_path = f"{output_directory}/{output_filename}"

English_df = pd.read_json(output_path, lines=True)

# Remove Chinese transcript from Taukdial
def remove_zh_language_rows(df):
    return df[df['Languages'] != 'zh']

def clean_diagnosis(df):
    # Remove specific diagnoses
    diagnoses_to_remove = ['Vascular', 'Memory', 'Aphasia', "Pick's", 'Other']
    df = df[~df['Diagnosis'].isin(diagnoses_to_remove)]
    # Remove rows with empty Diagnosis
    df = df[df['Diagnosis'].notna() & (df['Diagnosis'] != '')]
    
    # Rename diagnoses
    df['Diagnosis'] = df['Diagnosis'].replace({
        'Control': 'HC',
        'Conrol': 'HC',      # There is a single sample in Lu with the 'Conrol' typo
        'NC': 'HC',
        'H': 'HC',
        'AD': 'Dementia',
        'PossibleAD': 'Dementia',
        'ProbableAD': 'Dementia',
        'potential dementia': 'Dementia',
        'D': 'Dementia',
        "Alzheimer's": 'Dementia'
    })
    
    return df

def preprocess_text(text):
    text = re.sub(r'\b[A-Z]{3}\b', '', text)
    text = re.sub(r'xxx', '', text)
    text = re.sub(r'<[^>]*>', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.replace('PAR', '')
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\\x[0-9A-Za-z_]+\\x', '', text) 
    text = re.sub(r'\b\w+:\s*', '', text) 
    text = text.replace('\n', ' ')
    text = text.replace('→', '')
    text = text.replace('(', '').replace(')', '')
    text = re.sub(r'[\\+^"/„]', '', text)
    text = re.sub(r"[_']", '', text)
    text = text.replace('\t', ' ')
    text = re.sub(r'\[.*?\]', '', text)
    text = text.replace('&=laughs', '')
    text = text.replace('&=nods', '')
    text = text.replace('&=coughs', '')
    text = text.replace('&=snaps:tongue', '')
    text = text.replace('<', '').replace('>', '')
    text = text.replace('*', '').replace('&', '')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'([.,!?;:])\s+\1', r'\1', text)
    text = re.sub(r'(\.\s*){2,}', '.', text)
    if '.' in text:
        text = text.rsplit('.', 1)[0] + '.' 

    return text

def remove_short_transcripts(df, min_length=60):
    return df[df['Text_length'] > min_length]


def make_split(English_df, random_state=42, test_size=0.2):
    English_df = remove_zh_language_rows(English_df)
    English_df = clean_diagnosis(English_df)
    English_df["Text_interviewer_participant"] = English_df["Text_interviewer_participant"].apply(preprocess_text)


    English_df['Text_length'] = English_df['Text_interviewer_participant'].apply(len)

    English_df = remove_short_transcripts(English_df)

    # Split to train-test, excluding WLS samples from the test set
    WLS_df = English_df[English_df['Dataset'] == 'WLS']
    English_df = English_df[English_df['Dataset'] != 'WLS']
    train_en, test_en = train_test_split(English_df, test_size=test_size, stratify=English_df['Diagnosis'], random_state=random_state)
    train_en = pd.concat([train_en, WLS_df], ignore_index=True)

    # Save train and test datasets as JSONL
    train_en.to_json(output_directory + train_filename, orient="records", lines=True, force_ascii=False)
    test_en.to_json(output_directory + test_filename, orient="records", lines=True, force_ascii=False)



Combined JSONL file saved to: .tmp/English.jsonl


### Sparse Baseline (TF-IDF)

In [17]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import argparse

# parser = argparse.ArgumentParser()
# parser.add_argument('--test_language', required=True)
# parser.add_argument('--task', required=True)
# parser.add_argument('--translated', required=True) # use "yes", if you want to the analysis using the English translated data 

# args_slurm = parser.parse_args()

path_to_data_folder = ".tmp"
train_en = pd.read_json(path_to_data_folder + "/train_English.jsonl", lines=True)
test_en = pd.read_json(path_to_data_folder + "/test_English.jsonl", lines=True)
# train_spa = pd.read_json(path_to_data_folder + "/translated_train_df_spa.jsonl", lines=True)
# train_gr = pd.read_json(path_to_data_folder+"/translated_train_gr.jsonl", lines=True)
# train_cha = pd.read_json(path_to_data_folder + "/translated_train_cha.jsonl", lines=True)
# test_spa=pd.read_json(path_to_data_folder + "/translated_test_df_spa.jsonl", lines=True)
# test_gr= pd.read_json(path_to_data_folder + "/translated_test_gr.jsonl", lines=True)
# test_cha= pd.read_json(path_to_data_folder + "/translated_test_cha.jsonl", lines=True)

# Multi-lingual training and testing
# train_dfs = [train_en, train_gr, train_cha, train_spa]
# test_dfs = {
#     'en': test_en,
#     'gr': test_gr,
#     'cha': test_cha,
#     'spa': test_spa
# }

# Mono-lingual training and testing
train_dfs = [train_en]
test_dfs = {
    'en': test_en
}




# Add a column for translated text for English dataset

# if args_slurm.translated== "yes":
#     train_en['translated'] = train_en['Text_interviewer_participant']
#     test_en['translated'] = test_en['Text_interviewer_participant']

def classify_language_dataset_TFIDF(train_dfs, test_dfs, test_language, random_state=42, task=None, translated=None, classifiers=None, split_seed=0):
    train_combined = pd.concat(train_dfs, ignore_index=True)
    if any(df.equals(train_en) for df in train_dfs):
        train_combined['Diagnosis'] = train_combined['Diagnosis'].replace('AD', 'Dementia')

    if task == "binary":
        train_combined = train_combined[train_combined['Diagnosis'] != 'MCI']

    if translated == "yes":
         X_train = train_combined['translated']
    else:
        X_train = train_combined['Text_interviewer_participant']
    y_train = train_combined['Diagnosis']
    
    tfidf = TfidfVectorizer()
    X_train_tfidf = tfidf.fit_transform(X_train)
    
    test_df = test_dfs[test_language]
    if any(df.equals(train_en) for df in train_dfs):
         test_df['Diagnosis'] = test_df['Diagnosis'].replace('AD', 'Dementia')
    if task == "binary":
        test_df = test_df[test_df['Diagnosis'] != 'MCI']
    
    if translated == "yes":
        X_test = test_df['translated']
    else:
        X_test = test_df['Text_interviewer_participant']
    
    y_test = test_df['Diagnosis']
    
    X_test_tfidf = tfidf.transform(X_test)
    
    if classifiers is None:
        classifiers = {
            'Decision Tree': (DecisionTreeClassifier(random_state=random_state), {'max_depth': [10, 20, 30]}),
            'Random Forest': (RandomForestClassifier(random_state=random_state), {'n_estimators': [50, 100, 200]}),
            'Naive Bayes': (MultinomialNB(), {'alpha': [0.5, 1.0, 1.5]}),
            'SVM': (SVC(random_state=random_state), {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}),
            'Logistic Regression': (LogisticRegression(random_state=random_state), {'C': [0.1, 1, 10]})
        }
    
    results = []
    for name, (clf, params) in classifiers.items():
        grid_search = GridSearchCV(clf, params, cv=5, scoring='accuracy')
        grid_search.fit(X_train_tfidf, y_train)
        
        y_pred = grid_search.predict(X_test_tfidf)
        
        report = classification_report(y_test, y_pred, output_dict=True)
        
        result = {
            'classifier': name,
            'best_params': grid_search.best_params_,
            'accuracy': report['accuracy'],
            'uar': report['macro avg']['recall'],
            'f1': report['macro avg']['f1-score'],
            'test_language': test_language,
            'task': task,
            'translated': translated,
            'split_seed': split_seed
        }
        results.append(result)
        
        print(f"Classifier: {name}")
        print(f"Best Parameters: {grid_search.best_params_}")
        print(f"Test Set Language: {test_language}")
        print(f"Classification Metrics:")
        print(f"Accuracy: {report['accuracy']:.4f}, UAR: {report['macro avg']['recall']:.4f}, F1: {report['macro avg']['f1-score']:.4f}")
        print("\n")
    
    # Find the best result by accuracy
    best_result = max(results, key=lambda x: x['accuracy']) 
    
    return best_result

# classify_language_dataset_TFIDF(train_dfs, test_dfs, args_slurm.test_language, task=args_slurm.task,translated=args_slurm.translated)


### Dense Baseline (multilingual-e5-large)

In [18]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import GridSearchCV
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.svm import SVC
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import classification_report
# from sentence_transformers import SentenceTransformer
# import torch
# import argparse

# parser = argparse.ArgumentParser()
# parser.add_argument('--test_language', required=True)
# parser.add_argument('--task', required=True)
# parser.add_argument('--translated', required=True)

# args_slurm = parser.parse_args()

# path_to_data_folder = ".tmp"
# train_en = pd.read_json(path_to_data_folder + "/train_English.jsonl", lines=True)
# test_en = pd.read_json(path_to_data_folder + "/test_English.jsonl", lines=True)


# # train_spa = pd.read_json(path_to_data_folder + "/translated_train_df_spa.jsonl", lines=True)
# # train_gr = pd.read_json(path_to_data_folder+"/translated_train_gr.jsonl", lines=True)
# # train_cha = pd.read_json(path_to_data_folder + "/translated_train_cha.jsonl", lines=True)
# # test_spa=pd.read_json(path_to_data_folder + "/translated_test_df_spa.jsonl", lines=True)
# # test_gr= pd.read_json(path_to_data_folder + "/translated_test_gr.jsonl", lines=True)
# # test_cha= pd.read_json(path_to_data_folder + "/translated_test_cha.jsonl", lines=True)

# # Multi-lingual training and testing
# # train_dfs = [train_en, train_gr, train_cha, train_spa]
# # test_dfs = {
# #     'en': test_en,
# #     'gr': test_gr,
# #     'cha': test_cha,
# #     'spa': test_spa
# # }

# # Mono-lingual training and testing
# train_dfs = [train_en]
# test_dfs = {
#     'en': test_en
# }


# # Add a column for translated text for English dataset
# if args_slurm.translated== "yes":
#     train_en['translated'] = train_en['Text_interviewer_participant']
#     test_en['translated'] = test_en['Text_interviewer_participant']

# def extract_embeddings(df, text_column, label_column):
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'
#     model = SentenceTransformer('intfloat/multilingual-e5-large').to(device)
#     texts = ["passage: " + text for text in df[text_column].tolist()]
#     labels = df[label_column].tolist()
#     embeddings = model.encode(texts, normalize_embeddings=True,device=device)
#     return np.vstack(embeddings), np.array(labels)

# def classify_language_dataset_e5(train_dfs, test_dfs, test_language,random_state=42,task=None,translated=None):
#     # Combine the train sets from all languages
#     train_combined = pd.concat(train_dfs, ignore_index=True)
#     if any(df.equals(train_en) for df in train_dfs):
#         train_combined['Diagnosis'] = train_combined['Diagnosis'].replace('AD', 'Dementia')
#     if task== "binary":
#         train_combined = train_combined[train_combined['Diagnosis'] != 'MCI']
#     # Extract embeddings and labels for the combined train set
#     if translated == "yes":  
#         X_train, y_train = extract_embeddings(train_combined, 'translated', 'Diagnosis')
#     else:
#         X_train, y_train = extract_embeddings(train_combined, 'Text_interviewer_participant', 'Diagnosis')
    
    
#     # Select the appropriate test set based on the test_language argument

#     test_df = test_dfs[test_language]
#     if any(df.equals(train_en) for df in train_dfs):
#          test_df['Diagnosis'] = test_df['Diagnosis'].replace('AD', 'Dementia')
#     if task == "binary":
#         test_df = test_df[test_df['Diagnosis'] != 'MCI']
    
#     if translated == "yes":
#         X_test, y_test = extract_embeddings(test_df, 'translated', 'Diagnosis')
#     else:
#        X_test, y_test = extract_embeddings(test_df, 'Text_interviewer_participant', 'Diagnosis')
    
    
#     # Define classifiers and their hyperparameters for grid search
#     classifiers = {
#         'Decision Tree': (DecisionTreeClassifier(random_state=random_state), {'max_depth': [10, 20, 30]}),
#         'Random Forest': (RandomForestClassifier(random_state=random_state), {'n_estimators': [50, 100, 200]}),
#         'SVM': (SVC(random_state=random_state), {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}),
#         'Logistic Regression': (LogisticRegression(random_state=random_state), {'C': [0.1, 1, 10]})
#     }
    
#     # Perform grid search and classification
#     for name, (clf, params) in classifiers.items():
#         grid_search = GridSearchCV(clf, params, cv=5, scoring='accuracy')
#         grid_search.fit(X_train, y_train)
        
#         # Predict on the test set
#         y_pred = grid_search.predict(X_test)
        
#         # Print classification report
#         print(f"Classifier: {name}")
#         print(f"Best Parameters: {grid_search.best_params_}")
#         print(f"Test Set Language: {test_language}")
#         print(classification_report(y_test, y_pred))
#         print("\n")
#     print("test dataset: ", test_language)
#     for df in train_dfs:
#         df_name = [name for name, value in globals().items() if value is df][0]
#         print(f"DataFrame in training set: {df_name}")
#     print(task)
#     print("e5_large")
#     print("Translation status: ",translated)


# # classify_language_dataset_e5(train_dfs, test_dfs, args_slurm.test_language, task=args_slurm.task,translated=args_slurm.translated)


## Random Seeds on Test Set

In [19]:
seed_results = []
for seed in [42, 43, 44, 45, 46, 12345, 54321, 4242, 2026, 2468]:
    make_split(English_df, random_state=seed)

    path_to_data_folder = ".tmp" # Or output_directory if it's in scope
    train_en = pd.read_json(path_to_data_folder + "/train_English.jsonl", lines=True)
    test_en = pd.read_json(path_to_data_folder + "/test_English.jsonl", lines=True)
    
    train_dfs = [train_en]
    test_dfs = {'en': test_en}

    print(f"TF-IDF Classification on seed={seed}:")
    result = classify_language_dataset_TFIDF(train_dfs, test_dfs, "en", task="multiclass", translated="no", split_seed=seed)
    seed_results.append(result)
    print(f"Best model for seed {seed}: {result['classifier']}\n")

# Display results as a dataframe
results_df = pd.DataFrame(seed_results)

# Compute means and stds for numeric columns
mean_acc = results_df['accuracy'].mean()
mean_uar = results_df['uar'].mean()
mean_f1 = results_df['f1'].mean()
std_acc = results_df['accuracy'].std()
std_uar = results_df['uar'].std()
std_f1 = results_df['f1'].std()

# Create new rows for mean and std
mean_row = {'split_seed': 'mean', 'classifier': '', 'accuracy': mean_acc, 'uar': mean_uar, 'f1': mean_f1}
std_row = {'split_seed': 'std-dev', 'classifier': '', 'accuracy': std_acc, 'uar': std_uar, 'f1': std_f1}

# Append the new rows to the DataFrame
results_df = pd.concat([results_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

print("Summary of best models per seed:")
print(results_df[['split_seed', 'classifier', 'accuracy', 'uar', 'f1']])

TF-IDF Classification on seed=42:
Classifier: Decision Tree
Best Parameters: {'max_depth': 10}
Test Set Language: en
Classification Metrics:
Accuracy: 0.5375, UAR: 0.5523, F1: 0.5424


Classifier: Random Forest
Best Parameters: {'n_estimators': 200}
Test Set Language: en
Classification Metrics:
Accuracy: 0.5792, UAR: 0.5035, F1: 0.5162


Classifier: Naive Bayes
Best Parameters: {'alpha': 1.5}
Test Set Language: en
Classification Metrics:
Accuracy: 0.4667, UAR: 0.3333, F1: 0.2121




/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Classifier: SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6625, UAR: 0.6328, F1: 0.6462


Classifier: Logistic Regression
Best Parameters: {'C': 10}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6792, UAR: 0.6446, F1: 0.6627


Best model for seed 42: Logistic Regression

TF-IDF Classification on seed=43:
Classifier: Decision Tree
Best Parameters: {'max_depth': 10}
Test Set Language: en
Classification Metrics:
Accuracy: 0.5958, UAR: 0.5741, F1: 0.5804


Classifier: Random Forest
Best Parameters: {'n_estimators': 100}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6042, UAR: 0.5435, F1: 0.5624


Classifier: Naive Bayes
Best Parameters: {'alpha': 1.5}
Test Set Language: en
Classification Metrics:
Accuracy: 0.4667, UAR: 0.3333, F1: 0.2121




/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Classifier: SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6708, UAR: 0.6419, F1: 0.6479


Classifier: Logistic Regression
Best Parameters: {'C': 10}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6583, UAR: 0.6307, F1: 0.6368


Best model for seed 43: SVM

TF-IDF Classification on seed=44:
Classifier: Decision Tree
Best Parameters: {'max_depth': 10}
Test Set Language: en
Classification Metrics:
Accuracy: 0.5208, UAR: 0.4959, F1: 0.5028


Classifier: Random Forest
Best Parameters: {'n_estimators': 50}
Test Set Language: en
Classification Metrics:
Accuracy: 0.5792, UAR: 0.5111, F1: 0.5199


Classifier: Naive Bayes
Best Parameters: {'alpha': 1.5}
Test Set Language: en
Classification Metrics:
Accuracy: 0.4667, UAR: 0.3333, F1: 0.2121




...

Classifier: SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6792, UAR: 0.6406, F1: 0.6560


Classifier: Logistic Regression
Best Parameters: {'C': 10}
Test Set Language: en
Classification Metrics:
Accuracy: 0.7292, UAR: 0.6990, F1: 0.7101


Best model for seed 4242: Logistic Regression

TF-IDF Classification on seed=2026:
Classifier: Decision Tree
Best Parameters: {'max_depth': 20}
Test Set Language: en
Classification Metrics:
Accuracy: 0.4958, UAR: 0.4777, F1: 0.4833


Classifier: Random Forest
Best Parameters: {'n_estimators': 50}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6417, UAR: 0.5711, F1: 0.5899


Classifier: Naive Bayes
Best Parameters: {'alpha': 1.5}
Test Set Language: en
Classification Metrics:
Accuracy: 0.4667, UAR: 0.3333, F1: 0.2121




/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Classifier: SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6375, UAR: 0.6221, F1: 0.6268


Classifier: Logistic Regression
Best Parameters: {'C': 10}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6583, UAR: 0.6524, F1: 0.6526


Best model for seed 2026: Logistic Regression

TF-IDF Classification on seed=2468:
Classifier: Decision Tree
Best Parameters: {'max_depth': 10}
Test Set Language: en
Classification Metrics:
Accuracy: 0.5333, UAR: 0.4901, F1: 0.4863


Classifier: Random Forest
Best Parameters: {'n_estimators': 50}
Test Set Language: en
Classification Metrics:
Accuracy: 0.5750, UAR: 0.5101, F1: 0.5229


Classifier: Naive Bayes
Best Parameters: {'alpha': 1.0}
Test Set Language: en
Classification Metrics:
Accuracy: 0.4667, UAR: 0.3333, F1: 0.2121




/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mr/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Classifier: SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6792, UAR: 0.6434, F1: 0.6563


Classifier: Logistic Regression
Best Parameters: {'C': 10}
Test Set Language: en
Classification Metrics:
Accuracy: 0.7000, UAR: 0.6671, F1: 0.6801


Best model for seed 2468: Logistic Regression

Summary of best models per seed:
   split_seed           classifier  accuracy       uar        f1
0          42  Logistic Regression  0.679167  0.644587  0.662677
1          43                  SVM  0.670833  0.641926  0.647923
2          44  Logistic Regression  0.700000  0.671042  0.684547
3          45                  SVM  0.712500  0.688416  0.705567
4          46  Logistic Regression  0.725000  0.702300  0.715137
5       12345  Logistic Regression  0.662500  0.644256  0.653034
6       54321                  SVM  0.662500  0.627040  0.638875
7        4242  Logistic Regression  0.729167  0.698982  0.710136
8        2026  Logistic Regression  

#### Results:
On 2411 samples:
```
Summary of best models per seed:
   split_seed           classifier  accuracy       uar        f1
0          42  Logistic Regression  0.679167  0.644587  0.662677
1          43                  SVM  0.670833  0.641926  0.647923
2          44  Logistic Regression  0.700000  0.671042  0.684547
3          45                  SVM  0.712500  0.688416  0.705567
4          46  Logistic Regression  0.725000  0.702300  0.715137
5       12345  Logistic Regression  0.662500  0.644256  0.653034
6       54321                  SVM  0.662500  0.627040  0.638875
7        4242  Logistic Regression  0.729167  0.698982  0.710136
8        2026  Logistic Regression  0.658333  0.652366  0.652566
9        2468  Logistic Regression  0.700000  0.667063  0.680086
10       mean                       0.690000  0.663798  0.675055
11    std-dev                       0.026802  0.026036  0.028033
```
\
On 2469 samples:
```
Summary of best models per seed:
   split_seed           classifier  accuracy       uar        f1
0          42                  SVM  0.698413  0.658886  0.679174
1          43                  SVM  0.706349  0.672487  0.685754
2          44  Logistic Regression  0.706349  0.681762  0.693702
3          45  Logistic Regression  0.702381  0.668098  0.688406
4          46                  SVM  0.710317  0.683660  0.701392
5       12345  Logistic Regression  0.698413  0.682571  0.694864
6       54321  Logistic Regression  0.722222  0.704886  0.714828
...
8        2026  Logistic Regression  0.726190  0.703237  0.712393
9        2468                  SVM  0.690476  0.663243  0.672378
10       mean                       0.702778  0.675138  0.689164
11    std-dev                       0.016674  0.021400  0.019515
```

## 10-Fold Cross Validation

In [20]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
# Assuming classify_language_dataset_TFIDF and train_dfs are already in memory

classifiers = {'SVM': (SVC(random_state=42), {'C': [10], 'kernel': ['rbf']})}

train_full = pd.concat(train_dfs, ignore_index=True)

# 1. Separate WLS data so it never enters the validation folds
wls_mask = train_full['Dataset'] == 'WLS'
train_non_wls = train_full[~wls_mask].reset_index(drop=True)
train_wls = train_full[wls_mask].reset_index(drop=True)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_results = []

# 2. Stratify based only on the non-WLS diagnoses
for fold_num, (train_idx, val_idx) in enumerate(skf.split(train_non_wls, train_non_wls['Diagnosis']), start=1):
    
    # Base train and validation sets (no WLS yet)
    train_fold_base = train_non_wls.iloc[train_idx]
    val_fold = train_non_wls.iloc[val_idx].reset_index(drop=True)
    
    # 3. Append WLS exclusively to the training fold
    train_fold = pd.concat([train_fold_base, train_wls], ignore_index=True)

    result = classify_language_dataset_TFIDF(
        train_dfs=[train_fold],
        test_dfs={'en': val_fold},
        test_language='en',
        task='multiclass',
        translated='no',
        classifiers=classifiers,
        split_seed=fold_num
    )
    result['fold'] = fold_num
    cv_results.append(result)

    print(f"Fold {fold_num}: classifier={result['classifier']}, accuracy={result['accuracy']:.4f}, uar={result['uar']:.4f}, f1={result['f1']:.4f}")

cv_df = pd.DataFrame(cv_results)
print("\n10-fold CV summary")
print(cv_df[['fold', 'classifier', 'accuracy', 'uar', 'f1']])
print("mean accuracy:", cv_df['accuracy'].mean(), "±", cv_df['accuracy'].std())
print("mean uar:", cv_df['uar'].mean(), "±", cv_df['uar'].std())
print("mean f1:", cv_df['f1'].mean(), "±", cv_df['f1'].std())

Classifier: SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Test Set Language: en
Classification Metrics:
Accuracy: 0.7812, UAR: 0.7522, F1: 0.7662


Fold 1: classifier=SVM, accuracy=0.7812, uar=0.7522, f1=0.7662
Classifier: SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Test Set Language: en
Classification Metrics:
Accuracy: 0.7396, UAR: 0.7063, F1: 0.7300


Fold 2: classifier=SVM, accuracy=0.7396, uar=0.7063, f1=0.7300
Classifier: SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6354, UAR: 0.6037, F1: 0.6182


Fold 3: classifier=SVM, accuracy=0.6354, uar=0.6037, f1=0.6182
Classifier: SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6979, UAR: 0.6679, F1: 0.6832


Fold 4: classifier=SVM, accuracy=0.6979, uar=0.6679, f1=0.6832
Classifier: SVM
Best Parameters: {'C': 10, 'kernel': 'rbf'}
Test Set Language: en
Classification Metrics:
Accuracy: 0.6458, UAR: 0.6157, F1: 0.6121


#### Results:
On 2411 samples:
```
mean accuracy: 0.6854166666666667 ± 0.0550760557191621
mean uar: 0.6538886298886298 ± 0.05476201356275758
mean f1: 0.6655683633849966 ± 0.058844611234895094
```
\
On 2469 samples:
```
mean accuracy: 0.6929009900990099 ± 0.0554844482798103
mean uar: 0.6615657392253136 ± 0.05974541811282563
mean f1: 0.6771167465840661 ± 0.06022681737888036
```